# Parte A — EDA: Percepção dos Brasileiros sobre o Racismo no Brasil

**Pesquisa:** CESOP / IPEC / 04829  
**Período de campo:** 14 a 18 de abril de 2023  
**Amostra:** 2.000 entrevistas — eleitores de 16 anos ou mais  
**Margem de erro:** ± 2 p.p. (95% de confiança)

---

## Sumário

1. [Setup e Carregamento dos Dados](#1)
2. [Perfil da Amostra](#2)
3. [Análise 1 — Paradoxo da Consciência Racial](#3)
4. [Análise 2 — Distância Social do Racismo](#4)
5. [Análise 3 — Anatomia do Apoio às Cotas](#5)
6. [Análise 4 — Motor das Desigualdades: Raça ou Classe?](#6)
7. [Análise 5 — Definição Popular de Racismo (corrigida)](#7)
8. [Análise 6 — Percepção do Racismo Policial](#8)
9. [Correlação Externa 1 — PNAD: Rendimento e Informalidade por Raça](#9)
10. [Correlação Externa 2 — TSE 2022: Representatividade Racial dos Eleitos](#10)
11. [Correlação Externa 3 — CNJ: Composição Racial do Poder Judiciário](#11)
12. [PCA — Dimensões Latentes na Percepção do Racismo](#12)
13. [Conclusões](#13)

## 1. Setup e Carregamento dos Dados <a id='1'></a>

In [ ]:
# ============================================================
# INSTALAÇÃO DE DEPENDÊNCIAS
# ============================================================
!pip install pyreadstat prince --quiet

In [ ]:
# ============================================================
# DOWNLOAD DOS DADOS DO GITHUB
# Ajuste a URL para o seu repositório
# ============================================================
import urllib.request, os, zipfile

GITHUB_RAW = "https://raw.githubusercontent.com/SEU_USUARIO/IMT_CD_PROJETO_1/main/parte_a_pesquisa_opiniao/dados/"
GITHUB_EXT = "https://raw.githubusercontent.com/SEU_USUARIO/IMT_CD_PROJETO_1/main/parte_a_pesquisa_opiniao/dados/dados_externos/"

# Dados principais
arq_principal = "04829.SAV"
if not os.path.exists(arq_principal):
    urllib.request.urlretrieve(GITHUB_RAW + arq_principal, arq_principal)
    print(f"✓ {arq_principal} baixado.")
else:
    print(f"✓ {arq_principal} já existe.")

# Dados externos
externos = [
    "tabela10374.csv",   # PNAD rendimento por raça
    "tabela10372.csv",   # PNAD horas trabalhadas por raça
    "tabela10361.xlsx",  # PNAD informalidade por raça
    "consulta_cand_2022_BRASIL.csv",  # TSE candidatos 2022
    "cnj_composicao_racial.csv",      # CNJ composição racial
]

os.makedirs("dados_externos", exist_ok=True)
for arq in externos:
    caminho = f"dados_externos/{arq}"
    if not os.path.exists(caminho):
        try:
            urllib.request.urlretrieve(GITHUB_EXT + arq, caminho)
            print(f"✓ {arq} baixado.")
        except:
            print(f"⚠ {arq} não encontrado no GitHub — verifique o path.")
    else:
        print(f"✓ {arq} já existe.")

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import pyreadstat
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configurações visuais
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['figure.dpi'] = 130
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.titleweight'] = 'bold'

# Paleta de cores para raça/cor
COR_RACA = {
    'Branca':   '#4C72B0',
    'Preta':    '#2d2d2d',
    'Parda':    '#C4874F',
    'Amarela':  '#F5C518',
    'Indígena': '#2ca02c'
}

print('✓ Imports concluídos.')

In [ ]:
# ============================================================
# CARREGAMENTO E PRÉ-PROCESSAMENTO
# ============================================================
df_raw, meta = pyreadstat.read_sav('04829.SAV')
df = df_raw.copy()

# Raça/Cor
df['raca_label'] = df['RACA'].map({1:'Branca',2:'Preta',3:'Parda',4:'Amarela',5:'Indígena'})
df['raca_grupo'] = df['RACA'].map({1:'Branca',2:'Negra',3:'Negra',4:'Outras',5:'Outras'})

# Sexo
df['sexo_label'] = df['SEXO'].map({1:'Masculino',2:'Feminino'})

# Faixa etária
df['fxidade_label'] = df['FX_ID'].map({1:'16-24',2:'25-34',3:'35-44',4:'45-59',5:'60+'})

# Região
df['regiao_label'] = df['REGIAO'].map({1:'Norte',2:'Nordeste',3:'Sudeste',4:'Sul',5:'Centro-Oeste'})

# Posição política
df['politica_label'] = df['P_POLITICA'].map({1:'Esquerda',2:'Centro',3:'Direita',4:'Sem posição',99:None})

# Escolaridade agrupada
def escolaridade_grupo(x):
    if pd.isna(x): return None
    if x <= 7:   return 'Fundamental I'
    elif x <= 11: return 'Fundamental II'
    elif x <= 14: return 'Ensino Médio'
    elif x == 15: return 'Superior Inc.'
    elif x == 16: return 'Superior Comp.'
    return None

df['escol_grupo'] = df['ESCOLARIDADE'].apply(escolaridade_grupo)
ORDEM_ESCOL = ['Fundamental I','Fundamental II','Ensino Médio','Superior Inc.','Superior Comp.']

# Renda familiar
df['renda_label'] = df['REND2'].map({
    1:'Mais de 20 SM',2:'10-20 SM',3:'5-10 SM',
    4:'2-5 SM',5:'1-2 SM',6:'Até 1 SM',99:None
})
ORDEM_RENDA = ['Até 1 SM','1-2 SM','2-5 SM','5-10 SM','10-20 SM']

# UF sigla
MAPA_UF = {11:'RO',12:'AC',13:'AM',14:'RR',15:'PA',16:'AP',17:'TO',
            21:'MA',22:'PI',23:'CE',24:'RN',25:'PB',26:'PE',27:'AL',28:'SE',29:'BA',
            31:'MG',32:'ES',33:'RJ',35:'SP',
            41:'PR',42:'SC',43:'RS',
            50:'MS',51:'MT',52:'GO',53:'DF'}
df['uf_sigla'] = df['UF'].map(MAPA_UF)

# P1 motor das desigualdades
df['p1_label'] = df['P1'].map({
    1:'Raça/Cor/Etnia',2:'Classe Social',3:'Gênero/Sexo',
    4:'Local de Moradia',5:'Deficiência',6:'Orientação Sexual',
    7:'Local de Origem',97:'Nenhuma/Outras',99:None
})

print(f'✓ Dataset carregado: {df.shape[0]} entrevistas, {df.shape[1]} variáveis.')

# ============================================================
# HELPER: resposta múltipla cascateada (P7, P8, etc.)
# ============================================================
def escolheu_opcao_rm(df_in, opcao, prefixo, n_max=13):
    """Verifica se respondente escolheu a opção em qualquer posição da RM."""
    colunas = [f'{prefixo}_{i}' for i in range(1, n_max+1) if f'{prefixo}_{i}' in df_in.columns]
    mask = pd.Series(False, index=df_in.index)
    for col in colunas:
        mask = mask | (df_in[col] == float(opcao))
    return mask

# Binarizar P5 (Likert: 1=concordo totalmente, 2=concordo em parte → bin=1)
ITENS_P5 = {
    'P5A':'Brasil é racista',
    'P5B':'Eu sofri racismo',
    'P5C':'Tenho atitudes racistas',
    'P5D':'Presenciei racismo',
    'P5G':'Convivo c/ vítimas',
    'P5H':'Convivo c/ quem age racista',
    'P5I':'Minha família é racista',
}
for col in ITENS_P5:
    df[col+'_bin'] = df[col].apply(lambda x: 1 if x in [1,2] else (np.nan if x in [98,99] else 0))

print('✓ Pré-processamento concluído.')

---
## 2. Perfil da Amostra <a id='2'></a>

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Perfil da Amostra — CESOP/IPEC 04829\n2.000 entrevistas · Abril 2023', 
             fontsize=14, fontweight='bold', y=1.02)

variaveis = [
    ('sexo_label',    'Sexo',           ['#4C72B0','#DD8452']),
    ('raca_label',    'Raça/Cor',        list(COR_RACA.values())),
    ('fxidade_label', 'Faixa Etária',    sns.color_palette('Blues_d',5)),
    ('regiao_label',  'Região',          sns.color_palette('Set2',5)),
    ('escol_grupo',   'Escolaridade',    sns.color_palette('Purples_d',5)),
    ('politica_label','Posição Política', sns.color_palette('RdBu_r',4)),
]

for ax, (col, titulo, cores) in zip(axes.flat, variaveis):
    contagem = df[col].value_counts(dropna=True)
    pct = contagem.values / len(df) * 100
    bars = ax.barh(contagem.index, pct, color=cores[:len(contagem)], alpha=0.88)
    ax.set_title(titulo)
    ax.set_xlabel('%')
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0f}%'))
    for bar, val in zip(bars, pct):
        ax.text(val+0.4, bar.get_y()+bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=8.5)
    ax.set_xlim(0, max(pct)*1.18)

plt.tight_layout()
plt.savefig('fig_01_perfil_amostra.png', bbox_inches='tight', dpi=130)
plt.show()

---
## 3. Análise 1 — O Paradoxo da Consciência Racial <a id='3'></a>

> **Hipótese:** Existe gap sistemático entre reconhecer o racismo estrutural  
> ("O Brasil é racista") e assumir responsabilidade individual  
> ("Eu tenho atitudes racistas"). Esse gap varia por raça?

**Índice de Dissonância Cognitiva Racial (IDCR)** = concordância P5A − concordância P5C

In [ ]:
taxa_raca = df.groupby('raca_label')[['P5A_bin','P5C_bin']].mean() * 100
taxa_raca.columns = ['Brasil é racista (%)','Tenho atitudes racistas (%)']
taxa_raca['IDCR (gap p.p.)'] = taxa_raca['Brasil é racista (%)'] - taxa_raca['Tenho atitudes racistas (%)']
taxa_raca = taxa_raca.sort_values('IDCR (gap p.p.)', ascending=False)
print(taxa_raca.round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Paradoxo da Consciência Racial', fontsize=14, fontweight='bold')

# Painel esq: barras agrupadas
ax = axes[0]
x = np.arange(len(taxa_raca))
w = 0.35
b1 = ax.bar(x-w/2, taxa_raca['Brasil é racista (%)'], w,
             label='"O Brasil é racista" (P5A)', color='#c0392b', alpha=0.85)
b2 = ax.bar(x+w/2, taxa_raca['Tenho atitudes racistas (%)'], w,
             label='"Tenho atitudes racistas" (P5C)', color='#2980b9', alpha=0.85)
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(taxa_raca.index, rotation=15)
ax.set_ylabel('% que concorda (total ou em parte)')
ax.set_title('Concordância por Raça/Cor')
ax.legend(fontsize=9)
ax.set_ylim(0, 105)

# Painel dir: gap (IDCR)
ax2 = axes[1]
cores_gap = ['#e74c3c' if v>0 else '#3498db' for v in taxa_raca['IDCR (gap p.p.)']]
bars = ax2.barh(taxa_raca.index, taxa_raca['IDCR (gap p.p.)'], color=cores_gap, alpha=0.85)
ax2.axvline(0, color='black', linewidth=0.8, linestyle='--')
for bar, val in zip(bars, taxa_raca['IDCR (gap p.p.)']):
    ax2.text(val + (0.8 if val>=0 else -0.8),
             bar.get_y()+bar.get_height()/2,
             f'{val:.1f} p.p.', va='center', ha='left' if val>=0 else 'right', fontsize=9.5)
ax2.set_xlabel('Gap (p.p.): "Brasil racista" − "Tenho atitudes racistas"')
ax2.set_title('Índice de Dissonância Cognitiva Racial (IDCR)')
ax2.set_xlim(min(taxa_raca['IDCR (gap p.p.)'])-8, max(taxa_raca['IDCR (gap p.p.)'])+8)

plt.tight_layout()
plt.savefig('fig_02_paradoxo_consciencia.png', bbox_inches='tight', dpi=130)
plt.show()

In [ ]:
# Heatmap de todos os itens P5 por raça
cols_bin = [c+'_bin' for c in ITENS_P5]
labels_curtos = list(ITENS_P5.values())

matriz = df.groupby('raca_label')[cols_bin].mean() * 100
matriz.columns = labels_curtos

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(matriz, annot=True, fmt='.1f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, vmin=0, vmax=100,
            cbar_kws={'label':'% concordância'})
ax.set_title('% de Concordância com cada afirmação (P5) — por Raça/Cor', pad=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right', fontsize=9.5)
plt.tight_layout()
plt.savefig('fig_03_heatmap_p5.png', bbox_inches='tight', dpi=130)
plt.show()

# Teste qui-quadrado
from scipy.stats import chi2_contingency
tabela = pd.crosstab(df['raca_label'], df['P5C_bin'])
chi2, p, dof, _ = chi2_contingency(tabela)
print(f'Qui-quadrado (raça × atitudes racistas): χ²={chi2:.2f}, p={p:.4f}')
print('→ Diferença', 'SIGNIFICATIVA' if p<0.05 else 'NÃO significativa', '(p < 0.05)')

---
## 4. Análise 2 — Teoria da Distância Social do Racismo <a id='4'></a>

> **Hipótese:** A percepção do racismo declina conforme a afirmação se aproxima  
> da esfera pessoal. A **velocidade** desse declínio varia entre grupos raciais?

In [ ]:
ESCALA_DIST = [
    ('P5A_bin', '1. Brasil é\nracista'),
    ('P5D_bin', '2. Presenciei\nracismo'),
    ('P5G_bin', '3. Convivo c/\nvítimas'),
    ('P5I_bin', '4. Minha família\né racista'),
    ('P5B_bin', '5. Eu sofri\nracismo'),
    ('P5C_bin', '6. Tenho atitudes\nracistas'),
]
cols_d  = [c for c,_ in ESCALA_DIST]
labels_d = [l for _,l in ESCALA_DIST]

dist_raca = df.groupby('raca_label')[cols_d].mean() * 100
dist_raca.columns = labels_d

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Teoria da Distância Social do Racismo', fontsize=14, fontweight='bold')

# Painel esq: por raça
ax = axes[0]
for raca, row in dist_raca.iterrows():
    cor = COR_RACA.get(raca, '#999')
    ax.plot(range(len(labels_d)), row.values, marker='o', linewidth=2.2,
            markersize=7, color=cor, label=raca, alpha=0.9)
    # Rótulo no final com offset para não sobrepor
    ax.annotate(f'{raca} ({row.values[-1]:.0f}%)',
                xy=(len(labels_d)-1, row.values[-1]),
                xytext=(len(labels_d)-0.6, row.values[-1]),
                fontsize=8, color=cor, va='center')

ax.set_xticks(range(len(labels_d)))
ax.set_xticklabels(labels_d, rotation=0, ha='center', fontsize=9)
ax.set_ylabel('% que concorda (total ou em parte)')
ax.set_title('Por Raça/Cor')
ax.set_ylim(-5, 108)
ax.set_xlim(-0.3, len(labels_d)+0.5)
ax.axvspan(3.5, 5.5, alpha=0.07, color='red')
ax.axvspan(-0.3, 0.5, alpha=0.07, color='blue')
ax.text(4.5, 103, 'esfera pessoal', ha='center', fontsize=8, color='darkred', style='italic')
ax.text(0, 103, 'macro', ha='center', fontsize=8, color='darkblue', style='italic')
ax.grid(axis='y', alpha=0.4)

# Painel dir: por região
dist_reg = df.groupby('regiao_label')[cols_d].mean() * 100
dist_reg.columns = labels_d
ax2 = axes[1]
cores_reg = sns.color_palette('Set1', len(dist_reg))
for (reg, row), cor in zip(dist_reg.iterrows(), cores_reg):
    ax2.plot(range(len(labels_d)), row.values, marker='s', linewidth=2,
             markersize=6, color=cor, label=reg, alpha=0.85)
ax2.set_xticks(range(len(labels_d)))
ax2.set_xticklabels(labels_d, rotation=0, ha='center', fontsize=9)
ax2.set_ylabel('% que concorda')
ax2.set_title('Por Região')
ax2.set_ylim(-5, 108)
ax2.legend(title='Região', fontsize=8, loc='upper right')
ax2.grid(axis='y', alpha=0.4)

plt.tight_layout(pad=2.0)
plt.savefig('fig_04_distancia_social.png', bbox_inches='tight', dpi=130)
plt.show()

---
## 5. Análise 3 — Anatomia do Apoio às Cotas <a id='5'></a>

> **Hipótese:** O apoio às cotas segue lógica de grupo ou de ideologia política?  
> A hierarquia (PcD > Social > Racial > Mulheres > LGBTQIA+) revela algo sobre o caráter do apoio à inclusão?

In [ ]:
COTAS = {
    'P21A':'Cotas Geral',
    'P21B':'Cotas Raciais\n(negros/indígenas)',
    'P21C':'Cotas Sociais\n(baixa renda)',
    'P21D':'Cotas PcD',
    'P21E':'Cotas Mulheres',
    'P21F':'Cotas LGBTQIA+',
}

favor_geral = {label: (df[col]==1).sum()/df[col].notna().sum()*100
               for col, label in COTAS.items()}
favor_df = pd.Series(favor_geral).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Anatomia do Apoio às Cotas', fontsize=14, fontweight='bold')

# Painel esq: ranking
ax = axes[0]
cores = sns.color_palette('RdYlGn', len(favor_df))[::-1]
bars = ax.barh(favor_df.index, favor_df.values, color=cores, alpha=0.88)
ax.axvline(50, color='red', linestyle='--', linewidth=1.2, alpha=0.7, label='50%')
for bar, val in zip(bars, favor_df.values):
    ax.text(val+0.6, bar.get_y()+bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10)
ax.set_xlabel('% a favor')
ax.set_title('Ranking de Apoio por Tipo de Cota')
ax.set_xlim(0, 100)
ax.legend()

# Painel dir: cotas raciais × política × raça
ax2 = axes[1]
pivot = (df[df['politica_label'].notna() & df['raca_grupo'].notna()]
         .groupby(['politica_label','raca_grupo'])
         .apply(lambda g: (g['P21B']==1).sum()/g['P21B'].notna().sum()*100)
         .unstack())
ordem_pol = [p for p in ['Esquerda','Centro','Direita','Sem posição'] if p in pivot.index]
pivot = pivot.reindex(ordem_pol)
pivot.plot(kind='bar', ax=ax2, alpha=0.85, color=['#2d2d2d','#4C72B0','#C4874F'])
ax2.set_xlabel('Posição Política')
ax2.set_ylabel('% a favor de cotas raciais')
ax2.set_title('Apoio a Cotas Raciais:\nPosição Política × Raça')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=15)
ax2.axhline(50, color='red', linestyle='--', linewidth=1)
ax2.legend(title='Raça', fontsize=9)
ax2.set_ylim(0, 100)
for container in ax2.containers:
    ax2.bar_label(container, fmt='%.1f%%', fontsize=7.5, padding=2)

plt.tight_layout()
plt.savefig('fig_05_cotas.png', bbox_inches='tight', dpi=130)
plt.show()

---
## 6. Análise 4 — Motor das Desigualdades: Raça ou Classe? <a id='6'></a>

> **Pergunta:** Quem prefere "classe social" a "raça/cor" para explicar desigualdades?  
> Existe um **efeito de mascaramento de classe** entre respondentes de maior renda?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Motor Percebido das Desigualdades no Brasil (P01)', fontsize=14, fontweight='bold')

top_opcoes = ['Raça/Cor/Etnia','Classe Social','Local de Moradia','Gênero/Sexo']

# Painel esq: por raça
ct_raca = (pd.crosstab(df['raca_label'], df['p1_label'], normalize='index') * 100
           .reindex(columns=[c for c in top_opcoes if c in df['p1_label'].unique()], fill_value=0))
ct_raca.plot(kind='bar', ax=axes[0], alpha=0.85, color=sns.color_palette('Set2',4))
axes[0].set_xlabel('Raça/Cor')
axes[0].set_ylabel('%')
axes[0].set_title('Por Raça/Cor')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=15)
axes[0].legend(title='Fator principal', fontsize=8, loc='upper right')
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%.1f%%', fontsize=7, padding=1)

# Painel dir: raça vs classe por renda
ct_renda = (pd.crosstab(df['renda_label'], df['p1_label'], normalize='index') * 100)
cols_plot = [c for c in ['Raça/Cor/Etnia','Classe Social'] if c in ct_renda.columns]
ct_renda_plot = ct_renda[cols_plot].reindex([r for r in ORDEM_RENDA if r in ct_renda.index])
ct_renda_plot.plot(kind='bar', ax=axes[1], alpha=0.85, color=['#c0392b','#2980b9'])
axes[1].set_xlabel('Renda Familiar')
axes[1].set_ylabel('%')
axes[1].set_title('Raça vs. Classe Social por Faixa de Renda')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=20)
axes[1].legend(title='Fator', fontsize=9)
for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.1f%%', fontsize=8, padding=2)

plt.tight_layout()
plt.savefig('fig_06_motor_desigualdades.png', bbox_inches='tight', dpi=130)
plt.show()

---
## 7. Análise 5 — Definição Popular de Racismo (corrigida) <a id='7'></a>

> **Pergunta:** A população adota uma definição **estreita** (ação contra grupo racial)  
> ou **estrutural** de racismo? Isso prediz o apoio a políticas (P26)?

> ⚠️ **Nota metodológica:** P7 é resposta múltipla cascateada — cada respondente pode  
> indicar até 6 opções em sequência. A detecção correta verifica em TODAS as posições.

In [ ]:
P7_OPCOES = {
    1: 'Ação contra\ngrupo racial',
    2: 'Origem\nsocial/territorial',
    3: 'Práticas\nculturais',
    4: 'Religião',
    5: 'Características\npessoais',
    6: 'Produção de\ndesigualdades\n(estrutural)',
}

# Calcular taxa correta para cada opção
taxa_p7 = {}
for opcao, label in P7_OPCOES.items():
    sel = escolheu_opcao_rm(df, opcao, 'P7', n_max=6)
    # Excluir NS/NR (quem respondeu 99 em P7_1)
    validos = df['P7_1'] != 99
    taxa_p7[label] = sel[validos].mean() * 100

taxa_p7_s = pd.Series(taxa_p7).sort_values(ascending=True)
print('Taxa correta por opção:')
print(taxa_p7_s.round(1))

# Taxa de adoção da definição estrutural por escolaridade (corrigida)
df['escolheu_estrutural'] = escolheu_opcao_rm(df, 6, 'P7', n_max=6).astype(float)
df.loc[df['P7_1']==99, 'escolheu_estrutural'] = np.nan

taxa_estrut_escol = (df.groupby('escol_grupo')['escolheu_estrutural']
                     .mean() * 100
                     .reindex([e for e in ORDEM_ESCOL if e in df['escol_grupo'].unique()]))
print('\nTaxa def. estrutural por escolaridade:')
print(taxa_estrut_escol.round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Definição Popular de Racismo (P07)', fontsize=14, fontweight='bold')

# Painel esq
ax = axes[0]
cores_def = ['#e74c3c' if 'estrutural' in l else '#3498db' for l in taxa_p7_s.index]
bars = ax.barh(taxa_p7_s.index, taxa_p7_s.values, color=cores_def, alpha=0.85)
for bar, val in zip(bars, taxa_p7_s.values):
    ax.text(val+0.6, bar.get_y()+bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9.5)
ax.set_xlabel('% que selecionou esta definição')
ax.set_title('Frequência de cada Definição de Racismo')
ax.set_xlim(0, max(taxa_p7_s.values)*1.18)
red_p = mpatches.Patch(color='#e74c3c', label='Def. estrutural')
blue_p = mpatches.Patch(color='#3498db', label='Outras def.')
ax.legend(handles=[red_p, blue_p])

# Painel dir: def. estrutural por escolaridade (corrigido)
ax2 = axes[1]
escol_plot = [e for e in ORDEM_ESCOL if e in taxa_estrut_escol.index]
vals = taxa_estrut_escol.reindex(escol_plot).values
cores_e = sns.color_palette('Blues_d', len(escol_plot))
bars2 = ax2.bar(escol_plot, vals, color=cores_e, alpha=0.85)
for bar, val in zip(bars2, vals):
    if not np.isnan(val):
        ax2.text(bar.get_x()+bar.get_width()/2, val+0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_xlabel('Escolaridade')
ax2.set_ylabel('% que adota def. estrutural')
ax2.set_title('Adoção da Definição Estrutural\npor Escolaridade (corrigido)')
ax2.set_xticklabels(escol_plot, rotation=20, ha='right')
ax2.set_ylim(0, max(vals[~np.isnan(vals)])*1.25 if len(vals)>0 else 40)

plt.tight_layout()
plt.savefig('fig_07_definicao_racismo.png', bbox_inches='tight', dpi=130)
plt.show()

---
## 8. Análise 6 — Percepção do Racismo Policial e Institucional <a id='8'></a>

In [ ]:
P19_ITEMS = {
    'P19A':'Brancos e negros\ntratados diferente\npela polícia',
    'P19B':'Negros mais\ncriminalizados\ne punidos',
    'P19C':'Abordagem policial\nbaseada em cor/\ncabelo/vestimenta',
    'P19D':'Brasil tem políticas\npúblicas suficientes\np/ negros',
    'P19E':'Maior representatividade\nnegra reduz\ndesigualdades',
}

for col in P19_ITEMS:
    df[col+'_bin'] = df[col].apply(
        lambda x: 1 if x in [1,2] else (np.nan if x in [98,99] else 0))

taxa_p19_raca = df.groupby('raca_label')[[c+'_bin' for c in P19_ITEMS]].mean() * 100
taxa_p19_raca.columns = list(P19_ITEMS.values())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Percepção do Racismo Policial e Institucional (P19)', fontsize=14, fontweight='bold')

# Heatmap por raça
sns.heatmap(taxa_p19_raca, annot=True, fmt='.1f', cmap='RdYlGn',
            ax=axes[0], vmin=0, vmax=100, linewidths=0.5,
            cbar_kws={'label':'% concordância'})
axes[0].set_title('% Concordância por Raça/Cor')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right', fontsize=8)

# Por posição política
taxa_p19_pol = df.groupby('politica_label')[[c+'_bin' for c in P19_ITEMS]].mean() * 100
taxa_p19_pol.columns = [f'P19{c[-1]}' for c in P19_ITEMS]
taxa_p19_pol = taxa_p19_pol.reindex(
    [p for p in ['Esquerda','Centro','Direita','Sem posição'] if p in taxa_p19_pol.index])
taxa_p19_pol.T.plot(kind='bar', ax=axes[1], alpha=0.85, color=sns.color_palette('RdBu',4))
axes[1].set_xlabel('Item P19')
axes[1].set_ylabel('% que concorda')
axes[1].set_title('Percepção por Posição Política')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(title='Posição política', fontsize=8)
axes[1].set_ylim(0, 105)
axes[1].axhline(50, color='gray', linestyle='--', linewidth=0.8)
for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.0f%%', fontsize=6.5, padding=1)

plt.tight_layout()
plt.savefig('fig_08_racismo_policial.png', bbox_inches='tight', dpi=130)
plt.show()

---
## 9. Correlação Externa 1 — PNAD: Rendimento e Informalidade por Raça <a id='9'></a>

> **Fonte:** IBGE — PNAD Contínua Anual  
> **Tabela 10374:** Rendimento médio mensal real por cor/raça (2012–2025)  
> **Tabela 10361:** Informalidade e desocupação por cor/raça (2012–2025)  

> **Pergunta:** Estados/regiões com maior desigualdade salarial real entre brancos e negros  
> têm populações que **mais** ou **menos** percebem o Brasil como racista?

In [ ]:
# ============================================================
# LEITURA DAS TABELAS PNAD (SIDRA)
# ============================================================
def parse_sidra(fname, encoding='utf-8-sig', sep=';'):
    """Lê tabelas SIDRA com header multi-nível."""
    ext = fname.split('.')[-1].lower()
    if ext == 'xlsx':
        # Para XLSX, ler sheet específica
        return pd.read_excel(fname, sheet_name=0, header=None)
    
    with open(fname, encoding=encoding) as f:
        lines = f.readlines()
    
    anos_raw  = [x.strip().strip('"') for x in lines[3].split(sep)]
    racas_raw = [x.strip().strip('"') for x in lines[4].split(sep)]
    
    dados_raw = []
    for line in lines[5:]:
        if 'Fonte:' in line or line.strip() in ['','""']:
            break
        parts = [p.strip().strip('"').replace(',','.') for p in line.split(sep)]
        if parts and parts[0].strip():
            dados_raw.append(parts)
    
    if not dados_raw:
        return pd.DataFrame()
    
    n = len(dados_raw[0])
    ano_atual = ''
    col_names = []
    for i in range(n):
        a = anos_raw[i] if i < len(anos_raw) else ''
        r = racas_raw[i] if i < len(racas_raw) else ''
        if a.strip() and a.strip()[0].isdigit():
            ano_atual = a.strip()
        col_names.append(f'{ano_atual}_{r.strip()}' if i > 0 else 'local')
    
    df_out = pd.DataFrame(dados_raw, columns=col_names)
    for c in col_names[1:]:
        df_out[c] = pd.to_numeric(df_out[c], errors='coerce')
    return df_out

# Carregar rendimento
df_rend = parse_sidra('dados_externos/tabela10374.csv')
cols_rend_bpp = ['local'] + [c for c in df_rend.columns if c.endswith(('_Branca','_Preta','_Parda'))]
df_rend = df_rend[cols_rend_bpp]

print('Rendimento por raça — anos disponíveis:')
anos_disp = sorted(set(c.split('_')[0] for c in df_rend.columns[1:]))
print(anos_disp)
print()
print(df_rend[df_rend['local'].isin(['Brasil','Norte','Nordeste','Sudeste','Sul','Centro-Oeste'])].iloc[:,:10].to_string())

In [ ]:
# Gap salarial por região ao longo do tempo
regioes = ['Norte','Nordeste','Sudeste','Sul','Centro-Oeste']
anos_plot = [str(a) for a in range(2012, 2026) if f'{a}_Branca' in df_rend.columns]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Desigualdade Salarial por Raça — PNAD Contínua (2012–2025)', 
             fontsize=14, fontweight='bold')

# Painel esq: gap relativo Branca/Preta ao longo do tempo por região
ax = axes[0]
cores_reg = sns.color_palette('Set1', len(regioes))
for reg, cor in zip(regioes, cores_reg):
    row = df_rend[df_rend['local']==reg].iloc[0] if (df_rend['local']==reg).any() else None
    if row is None: continue
    gaps = []
    for ano in anos_plot:
        b = row.get(f'{ano}_Branca', np.nan)
        p = row.get(f'{ano}_Preta', np.nan)
        if pd.notna(b) and pd.notna(p) and p > 0:
            gaps.append((int(ano), (b/p - 1)*100))
    if gaps:
        x_vals, y_vals = zip(*gaps)
        ax.plot(x_vals, y_vals, marker='o', markersize=5, linewidth=2, 
                color=cor, label=reg, alpha=0.85)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Ano')
ax.set_ylabel('Gap (%): rendimento branco/preto − 1')
ax.set_title('Gap Salarial Branco/Preto por Região')
ax.legend(title='Região', fontsize=9)
ax.grid(alpha=0.3)
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

# Painel dir: comparação 2012 vs 2024 para Brasil e regiões  
ano_ini, ano_fim = '2012', '2024'
locais_plot = ['Brasil'] + regioes
dados_comp = []
for loc in locais_plot:
    row = df_rend[df_rend['local']==loc]
    if row.empty: continue
    row = row.iloc[0]
    for ano, nome_ano in [(ano_ini,'2012'),(ano_fim,'2024')]:
        b = row.get(f'{ano}_Branca', np.nan)
        p = row.get(f'{ano}_Preta', np.nan)
        pa = row.get(f'{ano}_Parda', np.nan)
        if pd.notna(b) and pd.notna(p):
            dados_comp.append({'local':loc,'ano':nome_ano,
                               'gap_preta':round((b/p-1)*100,1),
                               'gap_parda':round((b/pa-1)*100,1) if pd.notna(pa) else np.nan})

df_comp = pd.DataFrame(dados_comp)
x = np.arange(len(locais_plot))
w = 0.2
ax2 = axes[1]
for i, (ano, cor) in enumerate([('2012','#95a5a6'),('2024','#e74c3c')]):
    sub = df_comp[df_comp['ano']==ano].set_index('local').reindex(locais_plot)
    ax2.bar(x + i*w, sub['gap_preta'].values, w, label=f'Branca/Preta {ano}',
            color=cor, alpha=0.85)
ax2.set_xticks(x + w/2)
ax2.set_xticklabels(locais_plot, rotation=25, ha='right')
ax2.set_ylabel('Gap salarial (%)')
ax2.set_title('Gap Salarial Branco/Preto:\n2012 vs 2024')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)
for container in ax2.containers:
    ax2.bar_label(container, fmt='%.0f%%', fontsize=7.5, padding=2)

plt.tight_layout()
plt.savefig('fig_09_pnad_rendimento.png', bbox_inches='tight', dpi=130)
plt.show()

In [ ]:
# Correlação: percepção de racismo (CESOP) × gap salarial real (PNAD) por região
perc_regiao = df.groupby('regiao_label')['P5A_bin'].mean() * 100
perc_regiao = perc_regiao.reset_index()
perc_regiao.columns = ['regiao','perc_brasil_racista']

# Gap salarial por região no ano mais recente disponível
ano_recente = max(anos_plot, key=int)
gap_regiao = []
for reg in regioes:
    row = df_rend[df_rend['local']==reg]
    if row.empty: continue
    row = row.iloc[0]
    b = row.get(f'{ano_recente}_Branca', np.nan)
    p = row.get(f'{ano_recente}_Preta', np.nan)
    if pd.notna(b) and pd.notna(p) and p > 0:
        gap_regiao.append({'regiao':reg, 'gap_pct': round((b/p-1)*100, 1)})

df_gap_reg = pd.DataFrame(gap_regiao)
df_merge = perc_regiao.merge(df_gap_reg, on='regiao')

from scipy.stats import pearsonr
fig, ax = plt.subplots(figsize=(9, 6))
cores_r = sns.color_palette('Set1', len(df_merge))
for (_, row), cor in zip(df_merge.iterrows(), cores_r):
    ax.scatter(row['gap_pct'], row['perc_brasil_racista'], 
               color=cor, s=150, zorder=3, alpha=0.9)
    ax.annotate(row['regiao'], (row['gap_pct'], row['perc_brasil_racista']),
                textcoords='offset points', xytext=(8,3), fontsize=10, color=cor)

if len(df_merge) >= 3:
    r, p_val = pearsonr(df_merge['gap_pct'], df_merge['perc_brasil_racista'])
    z = np.polyfit(df_merge['gap_pct'], df_merge['perc_brasil_racista'], 1)
    x_line = np.linspace(df_merge['gap_pct'].min()-2, df_merge['gap_pct'].max()+2, 100)
    ax.plot(x_line, np.poly1d(z)(x_line), 'k--', alpha=0.5, linewidth=1.5)
    ax.text(0.05, 0.95, f'r = {r:.3f}  (p = {p_val:.3f})',
            transform=ax.transAxes, fontsize=11, va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))

ax.set_xlabel(f'Gap salarial branco/preto por região (%) — {ano_recente}\n(Fonte: PNAD Contínua / IBGE)')
ax.set_ylabel('% que concorda "O Brasil é um país racista" (P5A)\n(Fonte: CESOP/IPEC 2023)')
ax.set_title('Desigualdade Salarial Real vs. Percepção de Racismo por Região', fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig_10_pnad_correlacao.png', bbox_inches='tight', dpi=130)
plt.show()

---
## 10. Correlação Externa 2 — TSE 2022: Representatividade Racial dos Eleitos <a id='10'></a>

> **Fonte:** TSE — Repositório de Dados Eleitorais, Eleições 2022  
> **Pergunta:** A percepção de sub-representação do grupo étnico-racial (P20)  
> está calibrada com os dados reais de representatividade dos eleitos?

In [ ]:
# Carregar dados TSE
import zipfile

try:
    # Tentar ler o CSV direto
    df_tse = pd.read_csv('dados_externos/consulta_cand_2022_BRASIL.csv',
                         sep=';', encoding='latin1', low_memory=False)
except:
    print('⚠ Arquivo TSE não encontrado em dados_externos/')
    print('  Certifique-se de adicionar consulta_cand_2022_BRASIL.csv na pasta dados_externos/')
    df_tse = None

if df_tse is not None:
    # Filtrar válidos
    df_tse_val = df_tse[~df_tse['DS_COR_RACA'].isin(['#NULO','NÃO DIVULGÁVEL','NÃO INFORMADO'])].copy()
    df_tse_val = df_tse_val[df_tse_val['DS_COR_RACA'].notna()]
    
    # Cargos relevantes
    cargos = ['GOVERNADOR','SENADOR','DEPUTADO FEDERAL','DEPUTADO ESTADUAL']
    df_tse_rel = df_tse_val[df_tse_val['DS_CARGO'].isin(cargos)].copy()
    df_tse_rel['negro'] = df_tse_rel['DS_COR_RACA'].isin(['PRETA','PARDA'])
    df_tse_rel['eleito'] = df_tse_rel['DS_SIT_TOT_TURNO'].isin(['ELEITO','ELEITO POR QP','ELEITO POR MÉDIA'])
    
    # Taxa de eleição por raça e cargo
    taxa_tse = df_tse_rel.groupby(['DS_CARGO','DS_COR_RACA']).apply(
        lambda g: pd.Series({'candidatos':len(g),'eleitos':g['eleito'].sum(),
                             'taxa_%':g['eleito'].mean()*100})
    ).round(2)
    print('Taxa de eleição por raça e cargo:')
    print(taxa_tse[['candidatos','eleitos','taxa_%']].to_string())
    print(f'\nTotal entrevistados: {len(df_tse_rel):,}')

In [ ]:
if df_tse is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Representatividade Racial: TSE 2022 × Percepção CESOP', 
                 fontsize=14, fontweight='bold')

    # Painel esq: % negros candidatos vs eleitos por cargo
    ax = axes[0]
    cargos_plot = ['DEPUTADO ESTADUAL','DEPUTADO FEDERAL','SENADOR','GOVERNADOR']
    x = np.arange(len(cargos_plot))
    w = 0.3
    
    pct_cand_negro, pct_eleito_negro = [], []
    for cargo in cargos_plot:
        sub = df_tse_rel[df_tse_rel['DS_CARGO']==cargo]
        pct_cand_negro.append(sub['negro'].mean()*100)
        eleitos_cargo = sub[sub['eleito']]
        pct_eleito_negro.append(eleitos_cargo['negro'].mean()*100 if len(eleitos_cargo)>0 else 0)
    
    b1 = ax.bar(x-w/2, pct_cand_negro, w, label='% Negros entre candidatos',
                color='#95a5a6', alpha=0.85)
    b2 = ax.bar(x+w/2, pct_eleito_negro, w, label='% Negros entre eleitos',
                color='#e74c3c', alpha=0.85)
    ax.axhline(56.1, color='#2d2d2d', linestyle='--', linewidth=1.5, alpha=0.7,
               label='% Negros na pop. (IBGE 2022): 56,1%')
    ax.set_xticks(x)
    ax.set_xticklabels(cargos_plot, rotation=15, ha='right', fontsize=9)
    ax.set_ylabel('%')
    ax.set_title('Candidatos vs. Eleitos Negros por Cargo (TSE 2022)')
    ax.legend(fontsize=8)
    ax.set_ylim(0, 70)
    for container in [b1, b2]:
        ax.bar_label(container, fmt='%.1f%%', fontsize=8, padding=2)

    # Painel dir: percepção de não-representação (CESOP P20) por raça
    ax2 = axes[1]
    nao_rep_raca = (df[df['P20_1'].notna()]
                    .groupby('raca_label')
                    .apply(lambda g: (g['P20_1']==97).sum()/len(g)*100))
    nao_rep_geral = (df['P20_1']==97).sum() / df['P20_1'].notna().sum() * 100
    
    cores_bar = [COR_RACA.get(r,'#999') for r in nao_rep_raca.sort_values(ascending=False).index]
    bars = ax2.bar(nao_rep_raca.sort_values(ascending=False).index,
                   nao_rep_raca.sort_values(ascending=False).values,
                   color=cores_bar, alpha=0.85)
    ax2.axhline(nao_rep_geral, color='red', linestyle='--', linewidth=1.5,
                label=f'Média geral: {nao_rep_geral:.1f}%')
    for bar, val in zip(bars, nao_rep_raca.sort_values(ascending=False).values):
        ax2.text(bar.get_x()+bar.get_width()/2, val+0.3,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax2.set_ylabel('% que não se sente representado (P20)')
    ax2.set_title('Percepção de Não-Representação\npor Raça/Cor (CESOP 2023)')
    ax2.set_ylim(0, 35)
    ax2.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig('fig_11_tse_representacao.png', bbox_inches='tight', dpi=130)
    plt.show()

In [ ]:
if df_tse is not None:
    # Análise por UF: % negros eleitos para Dep. Federal × percepção CESOP
    dep_fed = df_tse_rel[df_tse_rel['DS_CARGO']=='DEPUTADO FEDERAL']
    uf_tse = dep_fed.groupby('SG_UF').apply(
        lambda g: pd.Series({
            'total_eleitos': g['eleito'].sum(),
            'negros_eleitos': (g['eleito'] & g['negro']).sum(),
            'pct_negros_eleitos': (g['eleito'] & g['negro']).sum()/g['eleito'].sum()*100
            if g['eleito'].sum()>0 else 0
        })
    ).reset_index()
    uf_tse.columns = ['uf_sigla','total_eleitos','negros_eleitos','pct_negros_eleitos']
    
    # Percepção de racismo por UF (CESOP)
    perc_uf = df.groupby('uf_sigla')['P5A_bin'].mean().reset_index()*100
    perc_uf.columns = ['uf_sigla','perc_racismo']
    # Corrigir multiplicação acidental
    perc_uf = df.groupby('uf_sigla')['P5A_bin'].mean().reset_index()
    perc_uf.columns = ['uf_sigla','perc_racismo']
    perc_uf['perc_racismo'] = perc_uf['perc_racismo'] * 100
    
    df_uf_merge = uf_tse.merge(perc_uf, on='uf_sigla')
    df_uf_merge = df_uf_merge[df_uf_merge['total_eleitos'] >= 5]  # UFs com amostra mínima
    
    from scipy.stats import pearsonr
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # Colorir por região
    mapa_uf_regiao = {
        'RO':'Norte','AC':'Norte','AM':'Norte','RR':'Norte','PA':'Norte','AP':'Norte','TO':'Norte',
        'MA':'Nordeste','PI':'Nordeste','CE':'Nordeste','RN':'Nordeste','PB':'Nordeste',
        'PE':'Nordeste','AL':'Nordeste','SE':'Nordeste','BA':'Nordeste',
        'MG':'Sudeste','ES':'Sudeste','RJ':'Sudeste','SP':'Sudeste',
        'PR':'Sul','SC':'Sul','RS':'Sul',
        'MS':'Centro-Oeste','MT':'Centro-Oeste','GO':'Centro-Oeste','DF':'Centro-Oeste'
    }
    cores_regiao_map = {'Norte':'#2ecc71','Nordeste':'#e74c3c','Sudeste':'#3498db',
                         'Sul':'#9b59b6','Centro-Oeste':'#f39c12'}
    
    for _, row in df_uf_merge.iterrows():
        reg = mapa_uf_regiao.get(row['uf_sigla'], 'Outros')
        cor = cores_regiao_map.get(reg, '#999')
        ax.scatter(row['pct_negros_eleitos'], row['perc_racismo'],
                   color=cor, s=90, zorder=3, alpha=0.85)
        ax.annotate(row['uf_sigla'], (row['pct_negros_eleitos'], row['perc_racismo']),
                    textcoords='offset points', xytext=(5,3), fontsize=8)
    
    if len(df_uf_merge) >= 3:
        r, p_val = pearsonr(df_uf_merge['pct_negros_eleitos'], df_uf_merge['perc_racismo'])
        z = np.polyfit(df_uf_merge['pct_negros_eleitos'], df_uf_merge['perc_racismo'], 1)
        x_line = np.linspace(df_uf_merge['pct_negros_eleitos'].min()-2,
                              df_uf_merge['pct_negros_eleitos'].max()+2, 100)
        ax.plot(x_line, np.poly1d(z)(x_line), 'k--', alpha=0.5, linewidth=1.5)
        ax.text(0.05, 0.95, f'r = {r:.3f}  (p = {p_val:.3f})',
                transform=ax.transAxes, fontsize=11, va='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))
    
    handles = [mpatches.Patch(color=c, label=r) for r,c in cores_regiao_map.items()]
    ax.legend(handles=handles, title='Região', fontsize=9)
    ax.set_xlabel('% de Deputados Federais Negros Eleitos por UF (TSE 2022)')
    ax.set_ylabel('% que concorda "O Brasil é racista" (P5A) — CESOP 2023')
    ax.set_title('Representatividade Real (TSE) vs. Percepção de Racismo (CESOP) por UF',
                  fontweight='bold')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('fig_12_tse_uf_scatter.png', bbox_inches='tight', dpi=130)
    plt.show()

---
## 11. Correlação Externa 3 — CNJ: Composição Racial do Poder Judiciário <a id='11'></a>

> **Fonte:** CNJ — Censo do Poder Judiciário 2021 + DataJud  
> **Pergunta:** A sub-representação negra no Judiciário segue o padrão  
> "pirâmide de exclusão" — piora quanto mais alto o cargo?

In [ ]:
# Dados do CNJ — composição racial por cargo
# Fonte: CNJ DataJud 2023 (arquivos xlsx carregados anteriormente)
cnj_dados = {
    'Servidores': {
        'Branca': 185732, 'Negro(a)-Pardo(a)': 68094, 
        'Negro(a)-Preto(a)': 12032, 'Amarelo(a)': 10007,
        'Indígena': 1349, 'Não declarado': 6794
    },
    'Juízes': {
        'Branca': 13169, 'Negro(a)-Pardo(a)': 2196,
        'Negro(a)-Preto(a)': 294, 'Amarelo(a)': 252,
        'Indígena': 25, 'Não declarado': 249
    },
    'Desembargadores': {
        'Branca': 2724, 'Negro(a)-Pardo(a)': 295,
        'Negro(a)-Preto(a)': 42, 'Amarelo(a)': 25,
        'Indígena': 1, 'Não declarado': 76
    }
}

df_cnj = pd.DataFrame(cnj_dados).T
df_cnj['Total'] = df_cnj.sum(axis=1)
df_cnj['% Negros (parda+preta)'] = (
    (df_cnj['Negro(a)-Pardo(a)'] + df_cnj['Negro(a)-Preto(a)']) / df_cnj['Total'] * 100
).round(1)
df_cnj['% Branca'] = (df_cnj['Branca'] / df_cnj['Total'] * 100).round(1)

print('=== Composição Racial do Poder Judiciário (CNJ 2023) ===')
print(df_cnj[['Total','% Branca','% Negros (parda+preta)']].to_string())
print()
print(f'Pop. negra no Brasil (IBGE 2022): ~56,1%')
print(f'Negros no Judiciário: {df_cnj.loc["Servidores","% Negros (parda+preta)"]:.1f}% (servidores) → {df_cnj.loc["Juízes","% Negros (parda+preta)"]:.1f}% (juízes) → {df_cnj.loc["Desembargadores","% Negros (parda+preta)"]:.1f}% (desembargadores)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Composição Racial do Poder Judiciário vs. Percepção da Sociedade', 
             fontsize=14, fontweight='bold')

# Painel esq: pirâmide de exclusão no Judiciário
ax = axes[0]
cargos_cnj = ['Servidores', 'Juízes', 'Desembargadores']
pct_negros = df_cnj.loc[cargos_cnj, '% Negros (parda+preta)'].values
pct_branca = df_cnj.loc[cargos_cnj, '% Branca'].values

x = np.arange(len(cargos_cnj))
w = 0.35
b1 = ax.bar(x-w/2, pct_branca, w, label='Brancos (%)', color='#4C72B0', alpha=0.85)
b2 = ax.bar(x+w/2, pct_negros, w, label='Negros — parda+preta (%)', color='#C4874F', alpha=0.85)
ax.axhline(56.1, color='#C4874F', linestyle='--', linewidth=2, alpha=0.6,
           label='% Negros na pop. BR (56,1%)')
ax.axhline(42.7, color='#4C72B0', linestyle='--', linewidth=2, alpha=0.6,
           label='% Brancos na pop. BR (42,7%)')
ax.set_xticks(x)
ax.set_xticklabels(cargos_cnj, fontsize=11)
ax.set_ylabel('%')
ax.set_title('Pirâmide de Exclusão Racial no Judiciário\n(CNJ 2023)')
ax.legend(fontsize=8, loc='upper right')
ax.set_ylim(0, 90)
for container in [b1, b2]:
    ax.bar_label(container, fmt='%.1f%%', fontsize=9, padding=3)

# Painel dir: percepção de não-representação no judiciário (P20)
ax2 = axes[1]
# P20: espaço 4 = Poder Judiciário
rep_jud = df.groupby('raca_label').apply(
    lambda g: pd.Series({
        'Sente representado': ((g['P20_1']==4) | (g['P20_2']==4) | 
                               (g['P20_3']==4)).mean()*100,
    })
)
cores_bar2 = [COR_RACA.get(r,'#999') for r in rep_jud.index]
bars2 = ax2.bar(rep_jud.index, rep_jud['Sente representado'],
                color=cores_bar2, alpha=0.85)
for bar, val in zip(bars2, rep_jud['Sente representado']):
    ax2.text(bar.get_x()+bar.get_width()/2, val+0.2,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_ylabel('% que sente seu grupo representado')
ax2.set_title('% que Sente Representação Adequada\nno Poder Judiciário (P20) — por Raça')
ax2.set_xticklabels(rep_jud.index, rotation=15)
ax2.set_ylim(0, 15)

plt.tight_layout()
plt.savefig('fig_13_cnj_representacao.png', bbox_inches='tight', dpi=130)
plt.show()

---
## 12. Análise Complementar — PCA nos Itens de Percepção (P5) <a id='12'></a>

> Identificar dimensões latentes na percepção do racismo.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

cols_pca = [c+'_bin' for c in ITENS_P5]
df_pca_data = df[cols_pca].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca_data)

pca = PCA(n_components=3)
componentes = pca.fit_transform(X_scaled)
variancia = pca.explained_variance_ratio_ * 100

print('Variância explicada:')
for i, v in enumerate(variancia):
    print(f'  PC{i+1}: {v:.1f}%')
print(f'  Total: {sum(variancia):.1f}%')

loadings = pd.DataFrame(
    pca.components_.T,
    index=list(ITENS_P5.values()),
    columns=[f'PC{i+1} ({v:.1f}%)' for i,v in enumerate(variancia)]
)
print('\nLoadings:')
print(loadings.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('PCA — Dimensões Latentes na Percepção do Racismo (P5)',
             fontsize=13, fontweight='bold')

# Scree plot
axes[0].bar(range(1,4), variancia, color='#3498db', alpha=0.85)
axes[0].plot(range(1,4), np.cumsum(variancia), 'ro-', linewidth=2, label='Cumulativo')
for i, v in enumerate(variancia):
    axes[0].text(i+1, v+0.3, f'{v:.1f}%', ha='center', fontsize=9)
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('Variância Explicada (%)')
axes[0].set_title('Scree Plot')
axes[0].legend()
axes[0].set_xticks(range(1,4))

# Biplot PC1 × PC2
df_idx = df.loc[df_pca_data.index].copy()
df_idx['PC1'] = componentes[:,0]
df_idx['PC2'] = componentes[:,1]

for raca, cor in COR_RACA.items():
    mask = df_idx['raca_label'] == raca
    axes[1].scatter(df_idx.loc[mask,'PC1'], df_idx.loc[mask,'PC2'],
                    color=cor, alpha=0.2, s=10, label=raca)

for nome, load in loadings.iterrows():
    axes[1].annotate('', xy=(load.iloc[0]*2.5, load.iloc[1]*2.5), xytext=(0,0),
                     arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    axes[1].text(load.iloc[0]*2.7, load.iloc[1]*2.7, nome, fontsize=7.5, ha='center')

axes[1].set_xlabel(f'PC1 ({variancia[0]:.1f}%)')
axes[1].set_ylabel(f'PC2 ({variancia[1]:.1f}%)')
axes[1].set_title('Biplot PC1 × PC2 por Raça/Cor')
axes[1].legend(markerscale=2, fontsize=8)
axes[1].axhline(0, color='gray', linewidth=0.5)
axes[1].axvline(0, color='gray', linewidth=0.5)

plt.tight_layout()
plt.savefig('fig_14_pca.png', bbox_inches='tight', dpi=130)
plt.show()

---
## 13. Conclusões <a id='13'></a>

### Principais Achados

| # | Análise | Achado Principal |
|---|---------|-----------------|
| 1 | **Paradoxo da Consciência Racial** | Gap sistemático entre reconhecer o racismo estrutural e assumir responsabilidade individual — mais pronunciado entre brancos (IDCR ~75 p.p.) |
| 2 | **Distância Social** | Percepção cai conforme se aproxima da esfera pessoal; velocidade do declínio varia por raça e região |
| 3 | **Cotas** | Cotas PcD têm mais apoio (88%) que cotas raciais (74%); apoio a cotas raciais cai acentuadamente à direita do espectro político |
| 4 | **Motor das Desigualdades** | Respondentes de maior renda tendem a atribuir desigualdades à "classe social" em vez de "raça" — efeito de mascaramento |
| 5 | **Definição de Racismo** | Apenas ~23% adota definição estrutural; adoção cresce levemente com escolaridade (18% → 27%) |
| 6 | **Racismo Policial** | Percepção alta mesmo entre brancos, mas varia fortemente por posição política |
| 7 | **PNAD** | Gap salarial branco/preto persiste em ~70% em todo o Brasil; não há correlação forte com percepção regional de racismo — paradoxo da normalização |
| 8 | **TSE 2022** | Negros são 51% dos candidatos mas apenas 32% dos eleitos; SC tem 0% de Dep. Federais negros |
| 9 | **CNJ** | Pirâmide de exclusão confirmada: 27% de negros entre servidores → 16% entre juízes → 11% entre desembargadores |

### Limitações
- Dados do CNJ são snapshot sem série histórica
- Pesquisa CESOP captura percepções declaradas (viés de desejabilidade social)
- Correlações regionais têm N pequeno (5 regiões)

### Referências
- IBGE (2023). PNAD Contínua. https://sidra.ibge.gov.br
- TSE (2022). Dados de Candidatos. https://dadosabertos.tse.jus.br
- CNJ (2023). DataJud / Censo do Poder Judiciário. https://www.cnj.jus.br
- CESOP/IPEC (2023). Pesquisa 04829 — Percepção dos Brasileiros sobre Racismo.

### Uso de IA
Este trabalho utilizou Claude (Anthropic) para suporte na estruturação do notebook, 
diagnóstico de bugs e sugestão de análises. Todo o código foi revisado e validado pela equipe.

In [ ]:
import glob
figs = sorted(glob.glob('fig_*.png'))
print(f'Figuras geradas: {len(figs)}')
for f in figs:
    print(f'  {f}')